# MongoDB Data Ingestion

**Student:** Vishal Singh  
**Database:** MongoDB Atlas (Free Tier)  
**Dataset:** All_Beauty category from Amazon Reviews 2023

This notebook loads product and review data into MongoDB Atlas and records ingestion time for Task 1 reporting.

## 1. Imports & Connection

In [1]:
import sys
import time
import platform
import psutil
from datetime import datetime
from datasets import load_dataset
import datasets
from tqdm import tqdm

sys.path.append('..')
from common.helper import connect_mongodb
from common.const import (
    MONGODB_URI, MONGODB_DATABASE,
    MONGODB_PRODUCTS_COLLECTION, MONGODB_REVIEWS_COLLECTION,
    MONGODB_BATCH_SIZE
)

# Silence HuggingFace warnings
datasets.logging.set_verbosity_error()

print("Imports done")

Imports done


## 2. System Specs

Record hardware specs for the Notion ingestion table.

In [2]:
# System specs for Notion table
cpu = platform.processor()
ram_gb = round(psutil.virtual_memory().total / (1024**3), 1)

print("=" * 50)
print("SYSTEM SPECS (copy into Notion)")
print("=" * 50)
print(f"CPU : {cpu}")
print(f"RAM : {ram_gb} GB")
print(f"OS  : {platform.system()} {platform.release()}")
print("=" * 50)
print("\nNOTE: Run speedtest.net to get your upload speed")

SYSTEM SPECS (copy into Notion)
CPU : Intel64 Family 6 Model 140 Stepping 2, GenuineIntel
RAM : 7.8 GB
OS  : Windows 11

NOTE: Run speedtest.net to get your upload speed


## 3. Connect to MongoDB

In [3]:
# Connect
db, client = connect_mongodb()
products_col = db[MONGODB_PRODUCTS_COLLECTION]
reviews_col  = db[MONGODB_REVIEWS_COLLECTION]

print(f"Connected to MongoDB")
print(f"Database   : {MONGODB_DATABASE}")
print(f"Batch size : {MONGODB_BATCH_SIZE}")
print(f"Products   : {products_col.count_documents({}):,} existing documents")
print(f"Reviews    : {reviews_col.count_documents({}):,} existing documents")

Connected to MongoDB
Database   : amazon_reviews
Batch size : 1000
Products   : 0 existing documents
Reviews    : 0 existing documents


## 4. Load Products

Loading `raw_meta_All_Beauty` from HuggingFace and inserting into MongoDB in batches.

In [4]:
print("[1/2] Loading Product Metadata from HuggingFace...")
dataset = load_dataset(
    "McAuley-Lab/Amazon-Reviews-2023",
    "raw_meta_All_Beauty",
    split="full",
    trust_remote_code=True
)
print(f"Total products to insert: {len(dataset):,}")

# ── Ingestion with timing ──────────────────────────────────────────
products_batch   = []
total_inserted   = 0
total_failed     = 0

products_start = time.time()  # START TIMER

for item in tqdm(dataset, desc="Inserting products"):
    product_doc = {
        'parent_asin'    : item['parent_asin'],
        'title'          : item['title'],
        'description'    : item['description'],
        'main_category'  : item['main_category'],
        'average_rating' : item['average_rating'],
        'rating_number'  : item['rating_number'],
        'price'          : item['price'],
        'features'       : item['features'],
        'store'          : item['store'],
        'details'        : item['details'],
    }
    products_batch.append(product_doc)

    if len(products_batch) >= MONGODB_BATCH_SIZE:
        try:
            products_col.insert_many(products_batch, ordered=False)
            total_inserted += len(products_batch)
        except Exception as e:
            total_failed += len(products_batch)
        products_batch = []

# Insert remaining
if products_batch:
    try:
        products_col.insert_many(products_batch, ordered=False)
        total_inserted += len(products_batch)
    except Exception as e:
        total_failed += len(products_batch)

products_end  = time.time()  # STOP TIMER
products_time = products_end - products_start

print(f"\n✓ Products inserted : {total_inserted:,}")
print(f"✗ Products failed   : {total_failed:,}")
print(f"⏱ Ingestion time    : {products_time:.2f}s ({products_time/60:.2f} mins)")

[1/2] Loading Product Metadata from HuggingFace...
Total products to insert: 112,590


Inserting products: 100%|██████████| 112590/112590 [09:44<00:00, 192.59it/s]



✓ Products inserted : 112,590
✗ Products failed   : 0
⏱ Ingestion time    : 586.90s (9.78 mins)


## 5. Load Reviews

In [5]:
print("[2/2] Loading Reviews from HuggingFace...")
dataset = load_dataset(
    "McAuley-Lab/Amazon-Reviews-2023",
    "raw_review_All_Beauty",
    trust_remote_code=True
)
print(f"Total reviews to insert: {len(dataset['full']):,}")

# ── Ingestion with timing ──────────────────────────────────────────
reviews_batch  = []
total_inserted = 0
total_failed   = 0

reviews_start = time.time()  # START TIMER

for item in tqdm(dataset['full'], desc="Inserting reviews"):
    review_doc = {
        'parent_asin'      : item['parent_asin'],
        'user_id'          : item['user_id'],
        'rating'           : item['rating'],
        'title'            : item['title'],
        'text'             : item['text'],
        'timestamp'        : item['timestamp'],
        'verified_purchase': item['verified_purchase'],
        'helpful_vote'     : item['helpful_vote'],
    }
    reviews_batch.append(review_doc)

    if len(reviews_batch) >= MONGODB_BATCH_SIZE:
        try:
            reviews_col.insert_many(reviews_batch, ordered=False)
            total_inserted += len(reviews_batch)
        except Exception as e:
            total_failed += len(reviews_batch)
        reviews_batch = []

# Insert remaining
if reviews_batch:
    try:
        reviews_col.insert_many(reviews_batch, ordered=False)
        total_inserted += len(reviews_batch)
    except Exception as e:
        total_failed += len(reviews_batch)

reviews_end  = time.time()  # STOP TIMER
reviews_time = reviews_end - reviews_start

print(f"\n✓ Reviews inserted : {total_inserted:,}")
print(f"✗ Reviews failed   : {total_failed:,}")
print(f"⏱ Ingestion time   : {reviews_time:.2f}s ({reviews_time/60:.2f} mins)")

[2/2] Loading Reviews from HuggingFace...


Total reviews to insert: 701,528


Inserting reviews: 100%|██████████| 701528/701528 [51:57<00:00, 225.00it/s]  



✓ Reviews inserted : 701,528
✗ Reviews failed   : 0
⏱ Ingestion time   : 3120.94s (52.02 mins)


## 6. Create Indexes

In [6]:
print("[3/3] Creating indexes...")

index_start = time.time()

try:
    # Products
    products_col.create_index('parent_asin', unique=True)
    print("✓ products → parent_asin (unique)")

    # Reviews
    reviews_col.create_index('parent_asin')
    print("✓ reviews → parent_asin")

    reviews_col.create_index('user_id')
    print("✓ reviews → user_id")

    reviews_col.create_index([('parent_asin', 1), ('timestamp', -1)])
    print("✓ reviews → (parent_asin, timestamp) compound")

    # Note: text index skipped — Atlas free tier storage limit exceeded
    print("⚠ Text index skipped — Atlas free tier storage limit (512MB)")

except Exception as e:
    print(f"⚠ Index error: {e}")

index_time = time.time() - index_start
print(f"\n⏱ Index creation time: {index_time:.2f}s")

[3/3] Creating indexes...
✓ products → parent_asin (unique)
✓ reviews → parent_asin
✓ reviews → user_id
✓ reviews → (parent_asin, timestamp) compound
⚠ Text index skipped — Atlas free tier storage limit (512MB)

⏱ Index creation time: 6.94s


## 7. Ingestion Summary

Copy these numbers into the Notion Ingestion Time table.

In [7]:
total_time = products_time + reviews_time

print("=" * 55)
print("INGESTION SUMMARY — Copy into Notion")
print("=" * 55)
print(f"Products ingestion : {products_time:.2f}s ({products_time/60:.2f} mins)")
print(f"Reviews ingestion  : {reviews_time:.2f}s ({reviews_time/60:.2f} mins)")
print(f"Total time         : {total_time:.2f}s ({total_time/60:.2f} mins)")
print(f"Batch size         : {MONGODB_BATCH_SIZE}")
print(f"CPU                : {cpu}")
print(f"RAM                : {ram_gb} GB")
print("=" * 55)
print("\nIndexing Strategies:")
print("  • products: parent_asin (unique)")
print("  • reviews:  parent_asin")
print("  • reviews:  user_id")
print("  • reviews:  (parent_asin, timestamp) compound")
print("=" * 55)
print(f"\nFinal document counts:")
print(f"  Products : {products_col.count_documents({}):,}")
print(f"  Reviews  : {reviews_col.count_documents({}):,}")

# Close connection
client.close()
print("\nConnection closed.")

INGESTION SUMMARY — Copy into Notion
Products ingestion : 586.90s (9.78 mins)
Reviews ingestion  : 3120.94s (52.02 mins)
Total time         : 3707.84s (61.80 mins)
Batch size         : 1000
CPU                : Intel64 Family 6 Model 140 Stepping 2, GenuineIntel
RAM                : 7.8 GB

Indexing Strategies:
  • products: parent_asin (unique)
  • reviews:  parent_asin
  • reviews:  user_id
  • reviews:  (parent_asin, timestamp) compound

Final document counts:
  Products : 112,590
  Reviews  : 701,528

Connection closed.
